# 18. Lecke: AI ügynökök biztonságosítása kriptográfiai bizonylatokkal

## Gyakorlati jegyzetfüzet

Ez a jegyzet négy feladatot mutat be:

1. **Írd alá az első bizonylatot** egy ügynök eszközhívás esetén, és ellenőrizd.
2. **Babráld meg a bizonylatot**, és nézd meg, hogy az ellenőrzés sikertelen lesz.
3. **Építs fel egy három bizonylatból álló láncot**, és erősítsd meg a lánc integritását.
4. **Csomagolj be egy Microsoft Agent Framework eszközhívást**, hogy minden művelet bizonylatot generáljon.

Minden kriptográfiai primitív jól karbantartott könyvtárakból van importálva (`pynacl` Ed25519-hez, `jcs` RFC 8785 kanonikus JSON-hoz, `hashlib` a Python szabványkönyvtárból SHA-256-hoz). Maga a bizonylat logika egyszerű Python, amit olvashatsz és módosíthatsz.

Futtasd a cellákat sorrendben. Minden szakasz rövid és önálló.


## Telepítés

Telepítsd a két függőséget. Mindkettő engedélyező licenccel rendelkezik (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Segéd eszközök

Ez a két segéd kezeli a base64url kódolást (kitöltés nélkül) és az SHA-256 hashelést tetszőleges objektumokon. Ezek segítenek abban, hogy a jegyzet többi része a nyugta logikájára összpontosítson.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## 1. szakasz: Írja alá az első nyugtáját

Képzelje el, hogy a **Contoso Travel** ügynöke éppen lekérdezte egy ügyfél számára a Sydneyből Los Angelesbe tartó járatokat. Ezt az eszközhívást szeretnénk aláírt nyugtaként rögzíteni, hogy a későbbi ellenőr anélkül ellenőrizhesse, hogy megbízzon bennünk.

### 1.1. lépés: Hozzon létre egy aláíró kulcsot

A termelési környezetben az ügynök aláíró kulcsa egy hardveres biztonsági modulban (HSM), Azure Key Vaultban vagy hasonló védett tárolóban lenne. Ebben a leckében egy friss kulcsot generálunk memóriában.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### 1.2. lépés: Az átvételi összetevő összeállítása

Az összetevő tartalmaz mindent, amit az átvételnek igazolnia kell: ki cselekedett, milyen eszközzel, milyen argumentumokkal, mi volt a visszatérés, milyen szabályzat alapján, és mikor. Az argumentumokat és az eredményt hash-eljük, ahelyett, hogy beágyaznánk őket, így az átvétel nem szivárogtat ki érzékeny tartalmat.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### 1.3 lépés: Írd alá és szereld össze a blokkot

Három lépés:

1. Kanonizáld a payloadot JCS használatával úgy, hogy két megvalósítás, amely ugyanazt a logikai blokkot állítja elő, bájt-azonos bájtokat produkáljon.
2. Hash-eld a kanonizált bájtokat SHA-256-tal.
3. Írd alá a hash-t Ed25519 privát kulccsal.

Az aláírás ezután csatolásra kerül az eredeti payloadhoz, hogy előállítsa a végleges blokkot.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### 1.4 lépés: A bizonylat ellenőrzése

Az ellenőrzés megfordítja a folyamatot. Eltávolítjuk az aláírást, újraszámítjuk a kanonikus hash-t, és ellenőrizzük az aláírást a bizonylatban található nyilvános kulccsal.

Egy audítornak, aki elvégzi ezt az ellenőrzést, nincs szüksége semmire tőlünk, csak magára a bizonylatra. Nem kell szolgáltatást hívnia, kulcskönyvtárat lekérdeznie, vagy megbíznia bárkiben.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Azt kell látnod, hogy `Receipt is valid: True`. Az ügynök előállította az első kriptográfiailag aláírt audit rekordját.


## 2. rész: Manipuláció a nyugtával

A nyugták lényege, hogy visszafordíthatatlan módon megváltoztathatatlanok legyenek. Bizonyítsuk be ezt.

Pontosan egy karaktert módosítunk a nyugtán, és megnézzük, hogy a hitelesítés sikertelen lesz.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Mi történt most?

Amikor megváltoztattuk a `policy_id`-t, a kanonikus bájtok megváltoztak. Az ezen bájtok SHA-256 hash értéke megváltozott. A korábbi hash fölött készült aláírás már nem egyezik az új hashel. Az ellenőrzés helyesen `False` értéket ad vissza.

Nincs mód arra, hogy a blokkon belül bármely mezőt módosítsunk úgy, hogy az továbbra is igazolja magát, hacsak a támadónak nincs meg a privát kulcs. Amíg a privát kulcs egy kulcstárolóban van, és a nyilvános kulcs publikus, a hamisítás elrejtése lehetetlen.

Próbáld ki magad: módosítsd a `tool_name` vagy `agent_id` vagy `timestamp` mezőt a fentebbi cellában, majd futtasd újra. Minden változtatás érvénytelen bizonylatot eredményez.


## Section 3: Blockcímkék összekapcsolása

Egyetlen blokk címke egy műveletet véd. A legtöbb ügynök sok műveletet végez. Az egész sorozat megváltoztatás-ellenállóvá tétele érdekében minden blokk címkét összekapcsolunk az előzővel úgy, hogy az előző blokk címke hash-értékét beillesztjük az új blokk címke adatrészébe.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Ha bárki eltávolít vagy átstrukturál egy blokk címkét, a lánc pontosan ott törik meg. Bármely későbbi blokk címke ellenőrzése sikertelen lesz, mert annak `previous_receipt_hash` értéke már nem egyezik meg az előző hash értékével.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Most törje meg a láncot azzal, hogy meghamisítja a középső bizonylatot, majd ellenőrizze újra. A meghamisított bizonylat sikertelen lesz az aláírás-ellenőrzésen, ÉS a következő bizonylat megbukik a lánckapcsolati ellenőrzésen (mert a `previous_receipt_hash` már nem egyezik a módosított középső bizonylat hash-értékével).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

A 0. nyugta továbbra is érvényes (nem módosították, és nincs elődje, amihez kapcsolódnia kellene). Az 1. nyugta aláírás-ellenőrzése meghiúsul, mert megváltoztattuk a `tool_args_hash` értékét. A 2. nyugta lánchivatkozás-ellenőrzése meghiúsul, mert a `previous_receipt_hash` a módosítatlan (eredeti) 1. nyugtára vonatkozott.

Még ha egy támadó újraaláírná is a módosított 1. nyugtát (amit a privát kulcs nélkül nem tehet meg), a 2. nyugta lánchivatkozásának eltérése továbbra is felfedné a hamisítást. A változtatás elrejtéséhez a támadónak minden átalakítás utáni nyugtát újra kellene aláírnia, amihez szüksége van a privát kulcs birtoklására.


## 4. fejezet: Egy ügynök eszköz hívásának körbecsomagolása átvételi elismervény aláírással

Egy éles telepítés során nem szeretnéd, hogy minden ügynökszerző emlékezzen a `make_receipt` hívására. Azt akarod, hogy az átvételi elismervény aláírása automatikus legyen minden eszköz meghívásakor.

Íme a legegyszerűbb minta: egy burkoló osztály, amely bármilyen hívható eszközfüggvényt átvesz, és egy átvételi elismervényt kibocsátó verzióját adja vissza. Ez alkalmazkodik bármilyen ügynökkerethez, beleértve a Microsoft Agent Framework-öt is (`agent_framework.azure`).

Ha nincs beállítva Azure AI Foundry projekted, az alábbi helyi tesztminta így is bemutatja a mintát.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Integráció a Microsoft Agent Framework-kel

A fenti `ReceiptedTool` burkoló keretfüggetlen. Ahhoz, hogy egy Microsoft Agent Framework-kel készült agentben használd, regisztráld az így becsomagolt függvényt eszközként. Egy vázlat (a mockot valódi Azure AI Foundry eszközregisztrációra kell cserélni):

```python
# Pókkód az integrációs forma bemutatásához.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="Ön egy Contoso Travel ügynök...",
#     tools=[receipted_lookup],   # a becsomagolt eszköz, nem a nyers függvény
# )
# response = agent.run("Keress járatokat Sydneyből Los Angelesbe júniusban.")
#
# # A futtatás után az összes, az ügynök által használt eszközhívás aláírt bizonylattal rendelkezik:
# audit_chain = receipted_lookup.receipts
```

Az agent keretrendszernek nem kell semmit tudnia a nyugtákról. A nyugta aláírása az eszköz köré van csomagolva, nem beépítve a keretrendszerbe. Így adhatsz eredetiséget a meglévő agent kódhoz anélkül, hogy újra kellene írni az agentet.


## Összefoglaló és kiterjesztett kihívás

Ön:

- Létrehozott egy Ed25519 kulcspárt.
- Összeállított és aláírt egy bizonylatot egy ügynök eszköz hívásához.
- Ellenőrizte a bizonylatot offline módban csak a nyilvános kulccsal.
- Megbabrált egy bizonylattal és megfigyelte az érvényesítés sikertelenségét.
- Összeállított egy három bizonylatból álló hash-láncolt sorozatot.
- Megbabrált a lánc közepén és megfigyelte mind az aláírási, mind a lánc-kapcsolati hibát.
- Automatikus bizonylat aláírással látta el egy eszköz függvényét.

**Kiterjesztett kihívás.** Bővítse ki a bizonylat sémáját egy `request_id` mezővel (egy UUID az elosztott nyomon követéshez). Frissítse a `make_receipt` függvényt, hogy tartalmazza ezt, és ellenőrizze, hogy a bizonylatok továbbra is végponttól végpontig ellenőrizhetők legyenek. Ezután módosítsa a mezőt az aláírás után, és erősítse meg, hogy az ellenőrzés sikertelen lesz. Ez arra kényszerít, hogy belülről megértse, hogyan járul hozzá az alakiság szerinti kódolás minden egyes bájtja az aláíráshoz.

**Fontos határvonal.** A bizonylatok három dolgot és csak három dolgot bizonyítanak: hozzárendelés (ez a kulcs írta alá ezt a tartalmat), sértetlenség (a tartalom nem változott az aláírás óta), és sorrend (ez a bizonylat később keletkezett, mint az a bizonylat). Nem bizonyítják, hogy az ügynök cselekedete helyes volt, hogy a `policy_id`-ben megnevezett szabályzatot ténylegesen kiértékelték, vagy hogy az ügynök minden szabályt követett. A bizonylatok alapot jelentenek. A kormányzás az a rendszer, amit erre épít.

Olvassa újra a lecke README-jét ezzel a határvonallal a fejében. A leggyakoribb hiba, amit a csapatok a bizonylatokkal kapcsolatban elkövetnek, hogy azt feltételezik, „vannak bizonylataink” azt jelenti, „mi irányítva vagyunk.” Ez nem igaz. A bizonylatok átláthatóvá teszik az ügynök viselkedését. Nem teszik helyessé azt.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
